# Telco Customer Churn Model
Train four customer clusters and an L2-regularized logistic regression classifier.

In [ ]:
!pip install scikit-learn==1.8.0

In [ ]:
from pathlib import Path
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.base import clone
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, confusion_matrix,
                             classification_report, PrecisionRecallDisplay, RocCurveDisplay,
                             silhouette_score)
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.calibration import CalibrationDisplay
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
DATA_PATH = Path('Telco_customer_churn.xlsx')
OUTPUT_DIR = Path('evaluation')
OUTPUT_DIR.mkdir(exist_ok=True)
assert DATA_PATH.exists(), f'Missing original labeled data: {DATA_PATH.resolve()}'
raw = pd.read_excel(DATA_PATH)
print(f'Loaded {raw.shape[0]:,} customers and {raw.shape[1]} columns')
display(raw.head(3))

## Clean data and define target
Geographic fields, identifiers, duplicate/post-outcome churn fields, and the target are excluded from predictors.

In [ ]:
TARGET = 'Churn Value'
DROP_COLUMNS = ['Country', 'State', 'City', 'Zip Code', 'Lat Long', 'Latitude',
                'Longitude', 'CustomerID', 'Count', 'Churn Label', 'Churn Score',
                'Churn Reason', TARGET]
NUMERIC_EXPECTED = ['Tenure Months', 'Monthly Charges', 'Total Charges', 'CLTV']
NONNEGATIVE_FIELDS = set(NUMERIC_EXPECTED)

if TARGET not in raw:
    raise ValueError('Workbook is missing the Churn Value target.')
if raw[TARGET].isna().any() or not raw[TARGET].isin([0, 1]).all():
    raise ValueError('Target labels must be complete and binary (0/1).')

df = raw.copy()
audit_rows = []
for col in NUMERIC_EXPECTED:
    if col not in df:
        raise ValueError(f'Missing required numeric predictor: {col}')
    original = df[col]
    blank = original.isna() | original.astype('string').str.strip().eq('').fillna(False)
    converted = pd.to_numeric(original.where(~blank), errors='coerce')
    invalid = (~blank & converted.isna()) | converted.lt(0).fillna(False)
    if col == 'Tenure Months':
        invalid |= converted.notna() & (converted % 1 != 0)
    audit_rows.append({'Column': col, 'Missing/blank': int(blank.sum()),
                       'Invalid': int(invalid.sum()), 'Action': 'Train-set median imputation'})
    df[col] = converted.mask(blank | invalid).astype(float)

X = df.drop(columns=[c for c in DROP_COLUMNS if c in df.columns]).copy()
y = df[TARGET].astype(int)
for col in X.select_dtypes(exclude='number').columns:
    is_missing = X[col].isna() | X[col].astype('string').str.strip().eq('').fillna(False)
    audit_rows.append({'Column': col, 'Missing/blank': int(is_missing.sum()),
                       'Invalid': 0, 'Action': 'Train-set most-frequent imputation'})
    X.loc[is_missing, col] = np.nan

quality_report = pd.DataFrame(audit_rows).sort_values(['Invalid','Missing/blank'], ascending=False)
display(quality_report)
quality_report.to_csv(OUTPUT_DIR / 'training_data_quality.csv', index=False)
print('Input columns:', X.columns.tolist())
print('Overall churn rate:', f'{y.mean():.1%}')
print('Important: dataset-derived categorical vocabularies are validated by the app at inference time.')

## Train/test split
The two requested CSV exports contain predictor columns only, without the target variable.

In [ ]:
X_dev, X_test, y_dev, y_test = train_test_split(X, y, test_size=0.20,
                                                    random_state=RANDOM_STATE, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_dev, y_dev, test_size=0.25,
                                                 random_state=RANDOM_STATE, stratify=y_dev)
print('Train:', X_train.shape, 'Validation:', X_val.shape, 'Test:', X_test.shape)
print('Churn rates:', {k: round(v.mean(), 4) for k, v in
                       [('Train',y_train), ('Validation',y_val), ('Test',y_test)]})
X_dev.to_csv('training_data.csv', index=False)
X_test.to_csv('test_data.csv', index=False)
# Keep these labels separate; do NOT upload them as predictor columns.
y_test.to_csv(OUTPUT_DIR / 'heldout_test_labels.csv', index=False)

## Impute, scale, and one-hot encode
Fit preprocessing on training data only; unknown future categories are handled safely.

In [ ]:
numeric_features = X_train.select_dtypes(include='number').columns.tolist()
categorical_features = [c for c in X_train.columns if c not in numeric_features]
numeric_pipeline = Pipeline([('imputer', SimpleImputer(strategy='median')),
                             ('scaler', StandardScaler())])
categorical_pipeline = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')),
                                 ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])
preprocessor = ColumnTransformer([('numeric', numeric_pipeline, numeric_features),
                                  ('categorical', categorical_pipeline, categorical_features)],
                                 remainder='drop')
X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)
print('Encoded feature count:', X_train_processed.shape[1])
assert np.isfinite(X_train_processed).all()

## Cluster customers into four groups
K-Means is fitted on training customers only; test customers are assigned to learned clusters.

In [ ]:
cluster_diagnostics = []
cluster_candidates = {}
for k in range(2, 9):
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    c = km.fit_predict(X_train_processed)
    cluster_candidates[k] = km
    counts = np.bincount(c, minlength=k)
    cluster_diagnostics.append({'K': k, 'Inertia': km.inertia_,
      'Silhouette (sample)': silhouette_score(X_train_processed, c, sample_size=min(1800,len(c)), random_state=RANDOM_STATE),
      'Smallest cluster': int(counts.min()), 'Largest cluster': int(counts.max())})
cluster_diagnostics = pd.DataFrame(cluster_diagnostics)
display(cluster_diagnostics.round(3))
cluster_diagnostics.to_csv(OUTPUT_DIR / 'cluster_diagnostics.csv', index=False)
fig, ax = plt.subplots(figsize=(7,4))
ax.plot(cluster_diagnostics['K'], cluster_diagnostics['Silhouette (sample)'], marker='o')
ax.set(xlabel='K', ylabel='Sampled silhouette score', title='Cluster compactness and separation')
plt.show()

In [ ]:
def score_metrics(y_true, probability, threshold=0.5):
    pred = (probability >= threshold).astype(int)
    return {'Threshold': float(threshold), 'Accuracy': accuracy_score(y_true, pred),
            'Precision': precision_score(y_true, pred, zero_division=0),
            'Recall': recall_score(y_true, pred, zero_division=0),
            'F1': f1_score(y_true, pred, zero_division=0),
            'ROC AUC': roc_auc_score(y_true, probability),
            'Average precision': average_precision_score(y_true, probability)}

comparison_rows = []
validation_probabilities = {}
models = {}
base_lr = LogisticRegression(penalty='l2', C=1.0, solver='lbfgs', max_iter=2000, random_state=RANDOM_STATE)
base_lr.fit(X_train_processed, y_train)
p = base_lr.predict_proba(X_val_processed)[:,1]
models['Logistic regression'] = base_lr
validation_probabilities['Logistic regression'] = p
comparison_rows.append({'Model':'Logistic regression', **score_metrics(y_val,p)})

forest = RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                                random_state=RANDOM_STATE, n_jobs=-1)
forest.fit(X_train_processed, y_train)
p = forest.predict_proba(X_val_processed)[:,1]
models['Random forest'] = forest
validation_probabilities['Random forest'] = p
comparison_rows.append({'Model':'Random forest', **score_metrics(y_val,p)})

hybrids = {}
for k, km in cluster_candidates.items():
    tr_c = km.predict(X_train_processed)
    va_c = km.predict(X_val_processed)
    train_matrix = np.column_stack((X_train_processed,np.eye(k)[tr_c]))
    val_matrix = np.column_stack((X_val_processed,np.eye(k)[va_c]))
    clf = LogisticRegression(penalty='l2', C=1.0, solver='lbfgs', max_iter=2000, random_state=RANDOM_STATE)
    clf.fit(train_matrix,y_train)
    name = f'K-Means + logistic (K={k})'
    hybrids[k] = clf
    models[name] = clf
    p = clf.predict_proba(val_matrix)[:,1]
    validation_probabilities[name] = p
    comparison_rows.append({'Model':name, **score_metrics(y_val,p)})

comparison = pd.DataFrame(comparison_rows).sort_values('Average precision',ascending=False)
display(comparison.round(3))
comparison.to_csv(OUTPUT_DIR / 'validation_model_comparison.csv',index=False)
print('The best benchmark is exploratory; the saved Streamlit model below uses K=4.')

In [ ]:
cv_rows = []
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
for fold, (tr_idx, va_idx) in enumerate(cv.split(X_dev, y_dev), start=1):
    Xf_train, Xf_val = X_dev.iloc[tr_idx], X_dev.iloc[va_idx]
    yf_train, yf_val = y_dev.iloc[tr_idx], y_dev.iloc[va_idx]
    prep_fold = clone(preprocessor)
    xt = prep_fold.fit_transform(Xf_train)
    xv = prep_fold.transform(Xf_val)
    lr_fold = LogisticRegression(C=1.0,max_iter=2000,random_state=RANDOM_STATE).fit(xt,yf_train)
    cv_rows.append({'Model':'Logistic regression','Fold':fold,
      'Average precision': average_precision_score(yf_val,lr_fold.predict_proba(xv)[:,1]),
      'ROC AUC':roc_auc_score(yf_val,lr_fold.predict_proba(xv)[:,1])})
    km_fold = KMeans(n_clusters=4,n_init=10,random_state=RANDOM_STATE)
    tr_groups = km_fold.fit_predict(xt)
    val_groups = km_fold.predict(xv)
    lr_hybrid = LogisticRegression(C=1.0,max_iter=2000,random_state=RANDOM_STATE).fit(
        np.column_stack([xt,np.eye(4)[tr_groups]]),yf_train)
    p = lr_hybrid.predict_proba(np.column_stack([xv,np.eye(4)[val_groups]]))[:,1]
    cv_rows.append({'Model':'K-Means + logistic (K=4)','Fold':fold,
      'Average precision':average_precision_score(yf_val,p),'ROC AUC':roc_auc_score(yf_val,p)})
cv_results = pd.DataFrame(cv_rows)
display(cv_results.groupby('Model')[['Average precision','ROC AUC']].agg(['mean','std']).round(3))
cv_results.to_csv(OUTPUT_DIR / 'cross_validation_folds.csv',index=False)

## Train the four-cluster logistic regression model
Fit the selected K=4 configuration on the training split and evaluate it on the validation split. The K comparison above is exploratory; keeping K=4 maintains the original app design.

In [ ]:
# Reuse the fitted K=4 clustering model from the diagnostic section.
CLUSTER_COUNT = 4
kmeans = cluster_candidates[CLUSTER_COUNT]
train_clusters = kmeans.predict(X_train_processed)
val_clusters = kmeans.predict(X_val_processed)
test_clusters = kmeans.predict(X_test_processed)

def add_cluster_indicators(encoded, cluster_ids, k=CLUSTER_COUNT):
    return np.column_stack([encoded, np.eye(k)[cluster_ids]])

X_train_model = add_cluster_indicators(X_train_processed, train_clusters)
X_val_model = add_cluster_indicators(X_val_processed, val_clusters)
X_test_model = add_cluster_indicators(X_test_processed, test_clusters)

classifier = LogisticRegression(
    penalty='l2', C=1.0, solver='lbfgs',
    max_iter=2000, random_state=RANDOM_STATE
)
classifier.fit(X_train_model, y_train)
print('Model inputs:', X_train_model.shape[1], '(',
      X_train_processed.shape[1], 'encoded features +', CLUSTER_COUNT, 'cluster indicators)')
print('Validation average precision:', round(average_precision_score(
    y_val, classifier.predict_proba(X_val_model)[:, 1]), 3))

## Select a classification threshold on validation data, then evaluate once on the held-out test set
Threshold selection uses the validation split, not the test split. F1 is the default criterion; actual retention decisions may require a different tradeoff.

In [ ]:
# Choose a threshold using ONLY validation predictions.
val_probability = classifier.predict_proba(X_val_model)[:, 1]
threshold_rows = [score_metrics(y_val, val_probability, t)
                  for t in np.round(np.arange(0.10, 0.901, 0.025), 3)]
threshold_table = pd.DataFrame(threshold_rows)
selected = threshold_table.sort_values(
    ['F1', 'Recall', 'Threshold'], ascending=[False, False, True]
).iloc[0]
THRESHOLD = float(selected['Threshold'])
display(threshold_table.sort_values('F1', ascending=False).head(10).round(3))
threshold_table.to_csv(OUTPUT_DIR / 'validation_thresholds.csv', index=False)
print(f'Selected validation F1 threshold: {THRESHOLD:.3f}')

# FINAL evaluation: do not retune after inspecting test metrics.
y_probability = classifier.predict_proba(X_test_model)[:, 1]
y_pred = (y_probability >= THRESHOLD).astype(int)
test_metrics = score_metrics(y_test, y_probability, THRESHOLD)
print('Held-out test metrics:')
display(pd.DataFrame([test_metrics]).round(3))
print('Confusion matrix (true labels: rows, predicted labels: columns):')
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, zero_division=0))
pd.DataFrame([test_metrics]).to_csv(OUTPUT_DIR / 'heldout_test_metrics.csv', index=False)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
PrecisionRecallDisplay.from_predictions(y_test, y_probability, ax=axes[0])
RocCurveDisplay.from_predictions(y_test, y_probability, ax=axes[1])
CalibrationDisplay.from_predictions(y_test, y_probability, n_bins=10, ax=axes[2])
axes[0].set_title('Held-out precision–recall')
axes[1].set_title('Held-out ROC')
axes[2].set_title('Held-out calibration')
plt.tight_layout()
plt.show()

## Understand the customer segments
Profile clusters using the training split and show observed training churn rates. Cluster numbers have no intrinsic business meaning; assign names only after examining these results.

In [ ]:
training_segments = X_train.copy()
training_segments['Customer Segment'] = [f'Cluster {c+1}' for c in train_clusters]
training_segments['Observed Churn'] = y_train.to_numpy()
segment_summary = training_segments.groupby('Customer Segment', observed=True).agg(
    Customers=('Observed Churn', 'size'),
    Observed_churn_rate=('Observed Churn', 'mean'),
    Average_tenure=('Tenure Months', 'mean'),
    Average_monthly_charges=('Monthly Charges', 'mean'),
    Average_CLTV=('CLTV', 'mean')
).sort_index()
display(segment_summary.round(3))
segment_summary.to_csv(OUTPUT_DIR / 'training_segment_profile.csv')

# Segment-level predictions on the held-out set: these are risk estimates,
# not actual churn rates.
test_segments = pd.DataFrame({
    'Customer Segment': [f'Cluster {c+1}' for c in test_clusters],
    'Predicted probability': y_probability,
    'Actual churn': y_test.to_numpy()
})
heldout_summary = test_segments.groupby('Customer Segment').agg(
    Customers=('Actual churn','size'),
    Average_predicted_risk=('Predicted probability','mean'),
    Observed_churn_rate=('Actual churn','mean')
)
display(heldout_summary.round(3))
heldout_summary.to_csv(OUTPUT_DIR / 'heldout_segment_summary.csv')

## Explain the logistic regression model
Coefficients describe associations with predictions after preprocessing, **not causal effects**. Categorical variables are one-hot encoded and numerical variables standardized.

In [ ]:
encoded_names = preprocessor.get_feature_names_out().tolist()
feature_names = encoded_names + [f'Cluster {k+1} indicator' for k in range(CLUSTER_COUNT)]
assert len(feature_names) == len(classifier.coef_[0])
coefficients = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': classifier.coef_[0],
}).sort_values('Coefficient', ascending=False)
display(coefficients.head(10))
display(coefficients.tail(10))
coefficients.to_csv(OUTPUT_DIR / 'logistic_coefficients.csv', index=False)
fig, ax = plt.subplots(figsize=(9, 6))
plot_coeffs = pd.concat([coefficients.head(7), coefficients.tail(7)]).sort_values('Coefficient')
ax.barh(plot_coeffs['Feature'], plot_coeffs['Coefficient'])
ax.set(xlabel='Logistic regression coefficient (log-odds)',
       title='Largest positive and negative coefficients')
plt.tight_layout()
plt.show()

## Save model.pkl
The artifact contains preprocessing, K-Means, logistic regression, and the expected feature schema.

In [ ]:
# Export the exact preprocessor, clusterer, classifier, feature order and
# validation-selected threshold used for held-out evaluation.
model_artifact = {
    'preprocessor': preprocessor,
    'kmeans': kmeans,
    'classifier': classifier,
    'feature_columns': X_train.columns.tolist(),
    'numeric_features': numeric_features,
    'categorical_features': categorical_features,
    'excluded_columns': DROP_COLUMNS,
    'target_column': TARGET,
    'cluster_count': CLUSTER_COUNT,
    'threshold': THRESHOLD,
}
with open('model.pkl', 'wb') as f:
    pickle.dump(model_artifact, f)
print('Successfully saved model.pkl with threshold:', THRESHOLD)

## Verify saved model

In [ ]:
# Reload and verify that the exported model reproduces held-out probabilities.
# Only load pickle files that you trust.
with open('model.pkl', 'rb') as f:
    loaded = pickle.load(f)
test_transformed = loaded['preprocessor'].transform(X_test.reindex(columns=loaded['feature_columns']))
test_clusters_loaded = loaded['kmeans'].predict(test_transformed)
test_matrix = np.column_stack([
    test_transformed, np.eye(loaded['cluster_count'])[test_clusters_loaded]
])
reloaded_probabilities = loaded['classifier'].predict_proba(test_matrix)[:, 1]
np.testing.assert_allclose(reloaded_probabilities, y_probability)
print('Artifact predictions match notebook predictions.')
print('Exported threshold:', loaded['threshold'])
print('First five probabilities:', np.round(reloaded_probabilities[:5], 3))
print('First five clusters:', test_clusters_loaded[:5] + 1)